Backbone --> Aktivite öğrendi
Sonra --> User öğrenmye zorlandı

Multi-Task Model ile
--Bu hareket ne,bu hareketi kim yaptı?

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

import numpy as np

In [2]:
X = np.load("../data/X.npy")
y_user = np.load("../data/y_user.npy")
y_act = np.load("../data/y_act.npy")

print("X:", X.shape)
print("Users:", len(np.unique(y_user)))
print("Activities:", len(np.unique(y_act)))

X: (3521, 300, 90)
Users: 8
Activities: 16


In [3]:

X_train, X_test, y_user_train, y_user_test, y_act_train, y_act_test = train_test_split(
    X, y_user, y_act,
    test_size=0.2,
    stratify=y_user,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (2816, 300, 90)
Test: (705, 300, 90)


In [4]:
#Backbone kuralım

input_layer = Input(shape=(300,90))

x = Conv1D(64,5,padding='same',activation='relu')(input_layer)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)

x = Conv1D(128,3,padding='same',activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)

x = Conv1D(256,3,padding='same',activation='relu')(x)
x = BatchNormalization()(x)

shared_feature = GlobalAveragePooling1D()(x)

In [5]:
#Activity Head
#Featureyi 16 hareket sınıfına zorla ayırıyor
activity_output = Dense(16,activation='softmax',name='activitiy_output')(shared_feature)

In [6]:
#User Embedding Head

embedding = Dense(128,activation=None)(shared_feature)
embedding = Lambda(lambda x: tf.math.l2_normalize(x,axis=1),name='embedding_output')(embedding)

In [7]:
#Modeli oluşturalım

model = Model(inputs=input_layer,
               outputs= [activity_output,embedding])

In [8]:
#Triplet Loss

def triplet_loss(anchor,positive,negative,margin=0.5):
    pos_dist = tf.reduce_sum(tf.square(anchor-positive),axis=1)
    neg_dist = tf.reduce_sum(tf.square(anchor-negative),axis=1)
    
    loss = tf.maximum(pos_dist-neg_dist+margin,0.0)
    return tf.reduce_mean(loss)
    

In [9]:
#Activity loss
activity_loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

In [10]:
#Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

In [11]:
@tf.function
def train_step(anchor, positive, negative, y_anchor):
    with tf.GradientTape() as tape:
        
        # Anchor forward
        activity_pred, emb_anchor = model(anchor, training=True)
        
        # Positive forward
        _, emb_positive = model(positive, training=True)
        
        # Negative forward
        _, emb_negative = model(negative, training=True)
        
        # Activity loss
        act_loss = activity_loss_fn(y_anchor, activity_pred)
        
        # Triplet loss
        tri_loss = triplet_loss(emb_anchor, emb_positive, emb_negative)
        
        # Total loss
        total_loss = act_loss + 0.5 * tri_loss
    
    grads = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    
    return total_loss, act_loss, tri_loss

In [12]:
def generate_triplet_batch(X, y_user, y_act, batch_size=32):
    anchors = []
    positives = []
    negatives = []
    anchor_labels = []

    unique_users = np.unique(y_user)

    for _ in range(batch_size):
        user = np.random.choice(unique_users)
        user_indices = np.where(y_user == user)[0]

        # Anchor & Positive
        a_idx, p_idx = np.random.choice(user_indices, 2, replace=False)

        # Negative
        neg_indices = np.where(y_user != user)[0]
        n_idx = np.random.choice(neg_indices)

        anchors.append(X[a_idx])
        positives.append(X[p_idx])
        negatives.append(X[n_idx])
        anchor_labels.append(y_act[a_idx])

    return (
        np.array(anchors),
        np.array(positives),
        np.array(negatives),
        np.array(anchor_labels)
    )

In [13]:
#Training Loop
EPOCHS = 15
BATCH_SIZE = 32
STEPS_PER_EPOCH = 50

for epoch in range(EPOCHS):
    total_loss_epoch = 0
    act_loss_epoch = 0
    tri_loss_epoch = 0

    for step in range(STEPS_PER_EPOCH):
        a, p, n, y_a = generate_triplet_batch(X_train, y_user_train, y_act_train, BATCH_SIZE)
        
        total_loss, act_loss, tri_loss = train_step(a, p, n, y_a)
        
        total_loss_epoch += total_loss.numpy()
        act_loss_epoch += act_loss.numpy()
        tri_loss_epoch += tri_loss.numpy()

    print(f"Epoch {epoch+1}")
    print("Total Loss:", total_loss_epoch / STEPS_PER_EPOCH)
    print("Activity Loss:", act_loss_epoch / STEPS_PER_EPOCH)
    print("Triplet Loss:", tri_loss_epoch / STEPS_PER_EPOCH)

Epoch 1
Total Loss: 2.6509485
Activity Loss: 2.5936997
Triplet Loss: 0.11449697
Epoch 2
Total Loss: 2.33359
Activity Loss: 2.2840528
Triplet Loss: 0.09907531
Epoch 3
Total Loss: 2.0317624
Activity Loss: 1.9952104
Triplet Loss: 0.073103115
Epoch 4
Total Loss: 1.8271121
Activity Loss: 1.7937006
Triplet Loss: 0.066823356
Epoch 5
Total Loss: 1.7328395
Activity Loss: 1.7014961
Triplet Loss: 0.06268692
Epoch 6
Total Loss: 1.5548178
Activity Loss: 1.5297139
Triplet Loss: 0.05020792
Epoch 7
Total Loss: 1.482235
Activity Loss: 1.4556938
Triplet Loss: 0.053082213
Epoch 8
Total Loss: 1.3845406
Activity Loss: 1.3628179
Triplet Loss: 0.04344462
Epoch 9
Total Loss: 1.26714
Activity Loss: 1.2469007
Triplet Loss: 0.04047855
Epoch 10
Total Loss: 1.1675842
Activity Loss: 1.147244
Triplet Loss: 0.040680464
Epoch 11
Total Loss: 1.1352015
Activity Loss: 1.1158623
Triplet Loss: 0.038679037
Epoch 12
Total Loss: 1.0775012
Activity Loss: 1.0549167
Triplet Loss: 0.045168933
Epoch 13
Total Loss: 1.0210712
Activi

In [14]:
pred, _ = model(X_test)
acc = np.mean(np.argmax(pred, axis=1) == y_act_test)
print("Activity Accuracy:", acc)

Activity Accuracy: 0.7078014184397163


In [15]:
# Embedding çıkar
_, embeddings_test = model(X_test)

# Gallery oluştur (train set'ten)
_, embeddings_train = model(X_train)

gallery = {}
for user in np.unique(y_user_train):
    user_embs = embeddings_train[y_user_train == user]
    gallery[user] = np.mean(user_embs, axis=0)

In [16]:
def identify(sample_embedding, gallery, threshold=0.8):
    sims = {user: np.dot(sample_embedding, gallery[user]) for user in gallery}
    best_user = max(sims, key=sims.get)
    
    if sims[best_user] > threshold:
        return best_user
    else:
        return -1

In [17]:
pred, _ = model(X_test)
acc = np.mean(np.argmax(pred, axis=1) == y_act_test)
print("Activity Accuracy:", acc)

Activity Accuracy: 0.7078014184397163


In [18]:
pred_act, embeddings_test = model(X_test)
_, embeddings_train = model(X_train)

In [19]:
gallery = {}

for user in np.unique(y_user_train):
    user_embs = embeddings_train[y_user_train == user]
    gallery[user] = np.mean(user_embs, axis=0)

In [20]:
def identify(sample_embedding, gallery):
    sims = {user: np.dot(sample_embedding, gallery[user]) for user in gallery}
    return max(sims, key=sims.get)

correct = 0
for i in range(len(embeddings_test)):
    pred_user = identify(embeddings_test[i], gallery)
    if pred_user == y_user_test[i]:
        correct += 1

print("User Identification Accuracy:", correct / len(embeddings_test))

User Identification Accuracy: 0.9546099290780142
